In [ ]:
#@title Install
MODEL_CHOICE = 'qwen3omni'   #@param ['qwen3omni','qwen25omni','gemma3n','phi4mm']

from google.colab import drive
drive.mount('/content/drive')

ENVS = {
 'qwen3omni':  ['ms-swift==4.4.2','transformers==4.57.3','datasets==3.6.0',
                'torchao>=0.16.0','bitsandbytes','qwen-omni-utils','soundfile','decord'],
 'qwen25omni': ['ms-swift==4.4.2','transformers==4.57.3','datasets==3.6.0',
                'torchao>=0.16.0','bitsandbytes','qwen-omni-utils','soundfile','decord'],
 'gemma3n':    ['ms-swift==4.4.2','transformers==4.57.3','datasets==3.6.0',
                'torchao>=0.16.0','bitsandbytes','timm','soundfile'],
 'phi4mm':     ['ms-swift==4.4.2','transformers==4.48.3','peft==0.13.2',
                'datasets==3.6.0','torchao>=0.16.0','bitsandbytes','backoff',
                'scipy','soundfile'],
}
assert MODEL_CHOICE in ENVS, f'unknown model: {MODEL_CHOICE}'
for spec in ENVS[MODEL_CHOICE]:
    print('installing', spec)
    !pip install -q "{spec}"
!pip install -q accelerate

import importlib.metadata as _md
print()
for _p in ('ms-swift','transformers','datasets','torchao','peft','tokenizers','torch'):
    try: print(f'  {_p:14}{_md.version(_p)}')
    except Exception: print(f'  {_p:14}not installed')
import json, os
_marker = '/content/.model_choice_installed'
if os.path.exists(_marker):
    _installed = json.load(open(_marker))['choice']
    if _installed != MODEL_CHOICE:
        raise SystemExit(f'the runtime has the environment for {_installed!r}; restart it and run this cell again')
json.dump({'choice': MODEL_CHOICE}, open('/content/.model_choice_installed','w'))

In [ ]:
#@title Configuration
MODEL_CHOICE      = 'qwen3omni'   #@param ['qwen3omni','qwen25omni','gemma3n','phi4mm']
CONTRASTIVE_RATIO = 60
SEED              = 0
MODALITY          = 'both'
BALANCE_TRAIN     = True
BLIND_RATIO       = 0.05

import os, sys, glob, json
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/bearing_protocol')
PROJECT = ROOT
DATA_DIR = ROOT/'data_pb_5khz'
RUNS_DIR = ROOT/'runs'; JSONL_DIR = ROOT/'jsonl'; CKPT_ROOT = ROOT/'ckpt'
EVAL_DIR = ROOT/'eval'; SUBSET_DIR = EVAL_DIR/'subsets'
MODEL_CACHE = ROOT/'model_cache'
for p in (JSONL_DIR, CKPT_ROOT, RUNS_DIR, EVAL_DIR, SUBSET_DIR, MODEL_CACHE): p.mkdir(parents=True, exist_ok=True)
os.environ['MODELSCOPE_CACHE'] = str(MODEL_CACHE)
assert (DATA_DIR/'data_manifest_v4.json').exists(), 'run 1_paderborn_prepare first'
assert (DATA_DIR/'spec_v4.json').exists(), 'run 1_paderborn_prepare first'

_marker = '/content/.model_choice_installed'
if os.path.exists(_marker):
    _installed = json.load(open(_marker))['choice']
    assert _installed == MODEL_CHOICE, f'environment installed for {_installed!r}'

MODELS = {
 'qwen3omni':  dict(tag='qwen3omni-30b', id='Qwen/Qwen3-Omni-30B-A3B-Instruct',
                    mtype='qwen3_omni_moe', attn=None,   hours=14.5),
 'qwen25omni': dict(tag='qwen25omni-7b', id='Qwen/Qwen2.5-Omni-7B',
                    mtype='qwen2_5_omni',  attn='sdpa', hours=1.2),
 'gemma3n':    dict(tag='gemma3n-e4b',   id='google/gemma-3n-E4B-it',
                    mtype='gemma3n',      attn='eager', hours=2),
 'phi4mm':     dict(tag='phi4mm',        id='LLM-Research/Phi-4-multimodal-instruct',
                    mtype='phi4_multimodal', attn='sdpa', hours=1.5),
}
M = MODELS[MODEL_CHOICE]
MODEL_ID, MODEL_TAG, ATTN_IMPL, MODEL_TYPE = M['id'], M['tag'], M['attn'], M['mtype']

DATASET     = 'PB'
SPEC_ID     = 'pb_5khz_inferno_pg03_v4'
TEMPLATE_ID = 'v22_measured_evidence_v2'

NUM_EPOCHS=2; BATCH_SIZE=1; GRAD_ACCUM=4; LR=1e-4
LORA_RANK=16; LORA_ALPHA=64; MAX_LENGTH=4096
SAVE_STEPS=100; SAVE_LIMIT=2
STFT_MAX_FREQ_LABEL = 5000

RUN_TAG = f'{DATASET}_{MODEL_TAG}_{MODALITY}_r{CONTRASTIVE_RATIO}_s{SEED}'
assert CONTRASTIVE_RATIO in (0, 30, 45, 60, 75, 100), 'not a ratio of the sweep'

RATIO   = CONTRASTIVE_RATIO
RUN_ID  = RUN_TAG
PER_CLASS=206
SUBSET_SEED=12345
MAX_NEW_TOKENS=320
INFER_EXTRA = '--max_batch_size 8' if MODEL_CHOICE in ('qwen3omni','qwen25omni') else ''
RUN_SUBSTITUTION=True
SUBST_SEED=999
CLASSES=['Healthy','Inner_Ring_Fault','Outer_Ring_Fault']

def rec_path(rid): return RUNS_DIR/f'{rid}.json'
def rec_load(rid):
    p=rec_path(rid)
    return json.load(open(p)) if p.exists() else None
def rec_update(rid, **kw):
    r=rec_load(rid) or {'run_id':rid}
    r.update(kw); rec_path(rid).write_text(json.dumps(r, indent=1, default=str))
    return r

print('RUN_ID  ', RUN_ID); print('model   ', MODEL_ID, f'(type {MODEL_TYPE}, attn {ATTN_IMPL or "default"})')
_out=CKPT_ROOT/f'{RUN_TAG}_{TEMPLATE_ID}'
_ck=[p for p in glob.glob(str(_out/'**'/'checkpoint-*'), recursive=True) if os.path.isdir(p)]
print(f'{_out}: {len(_ck)} checkpoint(s), training {"resumes" if _ck else "starts from scratch"}')

In [ ]:
#@title Check the environment
import subprocess, sys, re
import importlib.metadata as _md
from pathlib import Path

EXPECT = {
 'qwen3omni':  {'ms-swift':'4.4.2','transformers':'4.57.3'},
 'qwen25omni': {'ms-swift':'4.4.2','transformers':'4.57.3'},
 'gemma3n':    {'ms-swift':'4.4.2','transformers':'4.57.3'},
 'phi4mm':     {'ms-swift':'4.4.2','transformers':'4.48.3','peft':'0.13.2'},
}[MODEL_CHOICE]

bad=[]
for pkg, want in EXPECT.items():
    got=_md.version(pkg)
    ok = got==want
    print(f'  {pkg:14}{got:12} want {want:10} {"OK" if ok else "MISMATCH"}')
    if not ok: bad.append(f'{pkg} is {got}, expected {want}')

def vt(v): return tuple(int(x) for x in re.findall(r'\d+', v)[:3])
ta=_md.version('torchao')
print(f'  {"torchao":14}{ta:12} need >=0.16.0  '
      f'{"OK" if vt(ta)>=vt("0.16.0") else "TOO OLD"}')
if vt(ta)<vt('0.16.0'): bad.append(f'torchao is {ta}, need >=0.16.0')

p=subprocess.run([sys.executable,'-c','import swift, transformers, datasets, torchao; '
                  'print(swift.__version__)'], capture_output=True, text=True)
print(f'\n  swift imports  {"OK" if p.returncode==0 else "FAILED"}   '
      f'{(p.stdout or p.stderr).strip()[:120]}')
if p.returncode!=0: bad.append('swift cannot import its dependencies')

import swift
from pathlib import Path as _P
_src=_P(swift.__file__).parent
_types=set()
for _p in (_src/'model').rglob('*.py'):
    _types |= set(re.findall(r'(?:MLLM|LLM)?ModelType\.(\w+)',
                             _p.read_text(errors='ignore')))
print(f'\n  model_type {MODEL_TYPE!r} registered in this swift: '
      f'{MODEL_TYPE in _types}')
if MODEL_TYPE not in _types:
    bad.append(f'{MODEL_TYPE} is not a model_type in ms-swift '
               f'{_md.version("ms-swift")}')

try:
    from modelscope.hub.api import HubApi
    HubApi().get_model(MODEL_ID)
    print(f'  identifier {MODEL_ID!r} resolves on ModelScope')
except Exception as _e:
    print(f'  identifier {MODEL_ID!r} did NOT resolve: {type(_e).__name__}')

if bad:
    print('\n'+'\n'.join('*** '+x for x in bad))
    print(f'run the install cell again with MODEL_CHOICE={MODEL_CHOICE!r}')
    raise SystemExit('environment does not match the chosen model')
print('\nenvironment OK')

In [ ]:
#@title Prompts and targets
import json, random
from pathlib import Path
from collections import Counter

SPEC = json.load(open(DATA_DIR/'spec_v4.json'))
TH   = SPEC['thresholds']
CLASSES=['Healthy','Inner_Ring_Fault','Outer_Ring_Fault']
PRETTY={'Healthy':'healthy bearing','Inner_Ring_Fault':'inner ring fault',
        'Outer_Ring_Fault':'outer ring fault'}
ARTICLED={'Healthy':'a healthy bearing','Inner_Ring_Fault':'an inner ring fault',
          'Outer_Ring_Fault':'an outer ring fault'}

SYSTEM_PROMPT = (
 "You are an expert bearing vibration analyst. You are given an audio rendering "
 "of a bearing vibration signal and its STFT spectrogram (0-5000 Hz). A "
 "convolutional classifier may also supply a preliminary diagnosis. Examine the "
 "signal evidence and explain what it shows, stating whether it supports or "
 "contradicts any supplied diagnosis.")

def _bin(feature, value):
    lo, hi = TH[feature]
    return 'low' if value < lo else ('high' if value >= hi else 'mid')

CHAR = {'low' :"Only a small fraction of the envelope energy in the 20-500 Hz band "
                "falls at the bearing characteristic frequencies; most of it is broadband.",
        'mid' :"A moderate fraction of the envelope energy in the 20-500 Hz band falls "
                "at the bearing characteristic frequencies.",
        'high':"A large fraction of the envelope energy in the 20-500 Hz band is "
                "concentrated at the bearing characteristic frequencies."}
def OUTER_E(b,hz):
    return {'low' :f"There is little energy at the outer-race passing rate near {hz:.1f} Hz.",
            'mid' :f"There is moderate energy at the outer-race passing rate near {hz:.1f} Hz.",
            'high':f"Envelope energy is strongly concentrated at the outer-race passing "
                   f"rate near {hz:.1f} Hz."}[b]
def OUTER_H(b):
    return {'low' :"No harmonic banding is visible at twice and three times that rate.",
            'mid' :"Weak harmonic content is present at twice and three times that rate.",
            'high':"Clear harmonic banding is present at twice and three times "
                   "that rate."}[b]
def INNER_E(b,hz):
    return {'low' :f"There is little energy at the inner-race passing rate near {hz:.1f} Hz.",
            'mid' :f"There is moderate energy at the inner-race passing rate near {hz:.1f} Hz.",
            'high':f"Envelope energy is strongly concentrated at the inner-race passing "
                   f"rate near {hz:.1f} Hz."}[b]
def INNER_S(b,fs):
    return {'low' :"No shaft-rate sideband is present below that rate.",
            'mid' :"A weak shaft-rate sideband is present below that rate.",
            'high':f"A clear shaft-rate sideband appears below that rate, spaced by "
                   f"the shaft frequency of {fs:.2f} Hz."}[b]

def dominant_peak_sentence(f):
    dom, bpfo, bpfi = f['env_dominant_freq_hz'], f['bpfo_hz'], f['bpfi_hz']
    tol = max(0.08*max(bpfo,1e-6), 3.0)
    if abs(dom-bpfo) <= tol or abs(dom-2*bpfo) <= 2*tol:
        return f"The dominant envelope peak at {dom:.0f} Hz coincides with the outer-race rate."
    if abs(dom-bpfi) <= tol or abs(dom-2*bpfi) <= 2*tol:
        return f"The dominant envelope peak at {dom:.0f} Hz coincides with the inner-race rate."
    return (f"The dominant envelope peak at {dom:.0f} Hz coincides with neither "
            f"characteristic rate.")

def evidence_sentences(f):
    return [CHAR[_bin('env_char_energy_frac', f['env_char_energy_frac'])],
            OUTER_E(_bin('env_bpfo_ratio', f['env_bpfo_ratio']), f['bpfo_hz']),
            OUTER_H(_bin('env_bpfo_harmonic_ratio', f['env_bpfo_harmonic_ratio'])),
            INNER_E(_bin('env_bpfi_ratio', f['env_bpfi_ratio']), f['bpfi_hz']),
            INNER_S(_bin('env_bpfi_sideband_ratio', f['env_bpfi_sideband_ratio']),
                    f['f_shaft_hz']),
            dominant_peak_sentence(f)]

def build_explanation(true_label, f, rng):
    s = evidence_sentences(f)
    closing = {'Healthy':"Taken together, the signal evidence is consistent with a "
                         "healthy bearing.",
               'Inner_Ring_Fault':"Taken together, the signal evidence is consistent "
                         "with an inner ring fault.",
               'Outer_Ring_Fault':"Taken together, the signal evidence is consistent "
                         "with an outer ring fault."}[true_label]
    return ' '.join(s + [closing])

REQUIRED = {
 'Outer_Ring_Fault':[('env_bpfo_ratio','high'),('env_bpfo_harmonic_ratio','high')],
 'Inner_Ring_Fault':[('env_bpfi_ratio','high'),('env_bpfi_sideband_ratio','high')],
 'Healthy':[('env_char_energy_frac','low')],
}
PHRASE = {
 'env_bpfo_ratio'          : lambda f: f"concentrated energy at the outer-race rate near {f['bpfo_hz']:.1f} Hz",
 'env_bpfo_harmonic_ratio' : lambda f: "harmonic banding at twice and three times that rate",
 'env_bpfi_ratio'          : lambda f: f"concentrated energy at the inner-race rate near {f['bpfi_hz']:.1f} Hz",
 'env_bpfi_sideband_ratio' : lambda f: "a shaft-rate sideband below that rate",
 'env_char_energy_frac'    : lambda f: "most envelope energy spread across the band rather than concentrated at the characteristic frequencies",
}

def missing_evidence(wrong, true_label, f):
    art = {'Healthy':'A healthy bearing','Inner_Ring_Fault':'An inner ring fault',
           'Outer_Ring_Fault':'An outer-race defect'}[wrong]
    absent=[k for k,want in REQUIRED[wrong] if _bin(k, f[k]) != want]
    if absent:
        parts=[PHRASE[k](f) for k in absent]
        joined=parts[0] if len(parts)==1 else parts[0]+" with "+parts[1]
        return f"{art} would show {joined}; the envelope does not."
    present=[k for k,want in REQUIRED[true_label] if _bin(k, f[k]) == want]
    if present:
        return (f"{art} is not the better account here: {PHRESENT[present[0]](f)} "
                f"is what the envelope shows.")
    return (f"{art} is not supported by the evidence above, which is weak at "
            f"both characteristic rates.")

PHRESENT = {
 'env_bpfo_ratio'          : lambda f: f"concentrated energy at the outer-race rate near {f['bpfo_hz']:.1f} Hz",
 'env_bpfo_harmonic_ratio' : lambda f: "harmonic banding above the outer-race rate",
 'env_bpfi_ratio'          : lambda f: f"concentrated energy at the inner-race rate near {f['bpfi_hz']:.1f} Hz",
 'env_bpfi_sideband_ratio' : lambda f: "a shaft-rate sideband below the inner-race rate",
 'env_char_energy_frac'    : lambda f: "envelope energy spread across the band rather than concentrated at the characteristic frequencies",
}

def build_contradiction_explanation(true_label, wrong_label, f, rng):
    assert wrong_label != true_label, 'contrastive sample must supply a WRONG label'
    base = build_explanation(true_label, f, rng)
    return (base + f" However, the CNN classified this as {ARTICLED[wrong_label]}. "
            + missing_evidence(wrong_label, true_label, f))

def build_blind_observation(f, rng):
    return ' '.join(evidence_sentences(f))

def user_prompt(f, supplied, modality):
    freq = (f"At the current shaft speed of {f['speed_rpm']:.0f} RPM, "
            f"BPFO={f['bpfo_hz']:.1f} Hz and BPFI={f['bpfi_hz']:.1f} Hz.")
    parts=[]
    if modality in ('both','audio'):
        parts.append("<audio>Listen to this bearing vibration signal.</audio>")
    if modality in ('both','stft'):
        parts.append(f"<image>STFT spectrogram (0-{STFT_MAX_FREQ_LABEL} Hz) of the same "
                     f"signal. {freq}</image>")
    else:
        parts.append(freq)
    body="\n".join(parts)
    if supplied is None:
        return body + "\n\nDescribe what you observe in the audio and spectrogram."
    return (body + f"\n\nCNN classification: {PRETTY[supplied]}\n"
            "Examine the signal evidence and explain this diagnosis.")

_r=random.Random(0)
_demo=json.load(open(DATA_DIR/'data_manifest_v4.json'))['train'][0]
_wrong=_r.choice([x for x in CLASSES if x!=_demo['label']])
print(f"--- prompt (contrastive: true={_demo['label']}, CNN says {_wrong}) ---")
print(user_prompt(_demo['features'],_wrong,MODALITY))
print('\n--- target ---')
print('<explanation>'+build_contradiction_explanation(
      _demo['label'],_wrong,_demo['features'],_r)+'</explanation>')

In [ ]:
#@title Checks on prompts and targets
DIAG=['env_bpfo_ratio','env_bpfi_ratio','env_char_energy_frac','env_dominant_freq_hz',
      'env_bpfo_harmonic_ratio','env_bpfi_sideband_ratio','kurtosis','crest_factor','rms']
man=json.load(open(DATA_DIR/'data_manifest_v4.json'))
probe=man['train'][0]['features']
leaks=[]
for mod in ('both','audio','stft'):
    for sup in [None]+CLASSES:
        p=user_prompt(probe,sup,mod)
        for k in DIAG:
            if k in ('bpfo_hz','bpfi_hz'): continue
            tok=f"{probe[k]:.3f}"[:5]
            if len(tok)>3 and tok in p: leaks.append((mod,sup,k))
assert not leaks, f'PROMPT LEAKS: {sorted(set(leaks))}'
print('leakage check passed')

BAD=['stationary defect','defect that passes','indicating a defect']
viol=[t for r in man['train'] if r['label']=='Healthy'
      for t in [build_explanation('Healthy', r['features'], random.Random(2))]
      if any(b in t for b in BAD)]
assert not viol, f'{len(viol)} Healthy targets assert a defect mechanism'
print('coherence check passed\n')

rr=random.Random(1)
by={}
for r in man['train']:
    by.setdefault(r['label'],[]).append(build_explanation(r['label'],r['features'],rr))
print()
for cls in CLASSES:
    v=by[cls]; u=len(set(v))
    print(f'{cls:18} n={len(v):5}  unique explanations={u:5}  ({100*u/len(v):.1f}%)')
for cls in CLASSES:
    print(f'\n--- {cls} example ---'); print(by[cls][0][:330]+'...')

In [ ]:
#@title Training set
train_dir=DATA_DIR/'train'; test_dir=DATA_DIR/'test'
def paths(split, rec):
    d = train_dir if split=='train' else test_dir
    return str(d/rec['audio']), rec['image']

rng=random.Random(SEED)
recs=list(man['train'])
if BALANCE_TRAIN:
    byc={}
    for r in recs: byc.setdefault(r['label'],[]).append(r)
    n_min=min(len(v) for v in byc.values())
    for v in byc.values(): rng.shuffle(v)
    recs=[r for v in byc.values() for r in v[:n_min]]; rng.shuffle(recs)
    print(f'balanced to {n_min}/class -> {len(recs)}')

n=len(recs); n_blind=int(round(n*BLIND_RATIO))
n_contr=int(round((n-n_blind)*CONTRASTIVE_RATIO/100.0))
idx=list(range(n)); rng.shuffle(idx)
blind_idx=set(idx[:n_blind]); contr_idx=set(idx[n_blind:n_blind+n_contr])

rows=[]
for i,r in enumerate(recs):
    f=r['features']; lab=r['label']; ap,ip=paths('train',r)
    if i in blind_idx:
        prompt=user_prompt(f,None,MODALITY); expl=build_blind_observation(f,rng)
        kind,sup='blind',None
    elif i in contr_idx:
        wrong=rng.choice([x for x in CLASSES if x!=lab])
        prompt=user_prompt(f,wrong,MODALITY)
        expl=build_contradiction_explanation(lab,wrong,f,rng); kind,sup='contrastive',wrong
    else:
        prompt=user_prompt(f,lab,MODALITY)
        expl=build_explanation(lab,f,rng); kind,sup='confirm',lab
    row={'messages':[{'role':'system','content':SYSTEM_PROMPT},
                     {'role':'user','content':prompt},
                     {'role':'assistant','content':f'<explanation>{expl}</explanation>'}],
         'meta':{'bearing':r['bearing'],'label':lab,'supplied':sup,'kind':kind}}
    if MODALITY in ('both','audio'): row['audios']=[ap]
    if MODALITY in ('both','stft'):  row['images']=[ip]
    rows.append(row)

train_jsonl=JSONL_DIR/f'train_{RUN_TAG}_{TEMPLATE_ID}.jsonl'
with open(train_jsonl,'w') as fh:
    for row in rows: fh.write(json.dumps({k:v for k,v in row.items() if k!='meta'})+'\n')
print(f'wrote {train_jsonl.name}  n={len(rows)}')
print('kind :', dict(Counter(r['meta']['kind'] for r in rows)))
print('label:', dict(Counter(r['meta']['label'] for r in rows)))
nc=sum(r['meta']['kind']=='contrastive' for r in rows)
nb_=sum(r['meta']['kind']=='blind' for r in rows)
print(f'contrastive {100*nc/(len(rows)-nb_):.1f}% of labelled rows | blind {100*nb_/len(rows):.1f}%')

In [ ]:
#@title Evaluation sets
test_recs=list(man['test']); erng=random.Random(SEED+1000)
CONTRASTIVE_MAP={'Healthy':'Outer_Ring_Fault','Inner_Ring_Fault':'Outer_Ring_Fault',
                 'Outer_Ring_Fault':'Inner_Ring_Fault'}
def erow(r,sup,kind):
    f=r['features']; ap,ip=paths('test',r)
    row={'messages':[{'role':'system','content':SYSTEM_PROMPT},
                     {'role':'user','content':user_prompt(f,sup,MODALITY)}],
         'meta':{'bearing':r['bearing'],'label':r['label'],'supplied':sup,'kind':kind,
                 'features':{k:f[k] for k in
                    ['bpfo_hz','bpfi_hz','f_shaft_hz','env_bpfo_ratio','env_bpfi_ratio',
                     'env_bpfo_harmonic_ratio','env_bpfi_sideband_ratio',
                     'env_char_energy_frac','env_dominant_freq_hz']}}}
    if MODALITY in ('both','audio'): row['audios']=[ap]
    if MODALITY in ('both','stft'):  row['images']=[ip]
    return row

sets={'correct'    :[erow(r,r['label'],'confirm') for r in test_recs],
      'contrastive':[erow(r,CONTRASTIVE_MAP[r['label']],'reject') for r in test_recs],
      'blind'      :[erow(r,None,'blind') for r in test_recs]}
for nm,rws in sets.items():
    p=JSONL_DIR/f'eval_{nm}_{DATASET}_{MODALITY}_{TEMPLATE_ID}.jsonl'
    if p.exists() and sum(1 for _ in open(p)) == len(rws):
        print(f'{nm:12} {len(rws):5} -> {p.name}  (exists, kept)'); continue
    with open(p,'w') as fh:
        for row in rws: fh.write(json.dumps(row)+'\n')
    print(f'{nm:12} {len(rws):5} -> {p.name}')

In [ ]:
#@title Run record
import math
n_train=len(rows)
steps_per_epoch=math.ceil(n_train/(BATCH_SIZE*GRAD_ACCUM))
planned_steps=steps_per_epoch*NUM_EPOCHS
OUT_DIR=CKPT_ROOT/f'{RUN_TAG}_{TEMPLATE_ID}'
_rec=rec_load(RUN_ID)
ALREADY_TRAINED=bool(_rec and _rec.get('status')=='done' and _rec.get('ckpt')
                     and Path(_rec['ckpt']).exists() and _rec.get('steps_actual')==planned_steps)
if ALREADY_TRAINED:
    print(f'{RUN_ID} is already trained ({_rec["steps_actual"]} steps, {_rec["ckpt"]}).')
else:
    rec_update(RUN_ID, dataset=DATASET, model=MODEL_TAG, model_id=MODEL_ID, modality=MODALITY,
               ratio=CONTRASTIVE_RATIO, seed=SEED, spec=SPEC_ID, template=TEMPLATE_ID,
               n_train=n_train, planned_steps=planned_steps, epochs=NUM_EPOCHS, lr=LR,
               lora_rank=LORA_RANK, lora_alpha=LORA_ALPHA, balanced=BALANCE_TRAIN,
               blind_ratio=BLIND_RATIO, train_jsonl=str(train_jsonl), out_dir=str(OUT_DIR),
               policy='final checkpoint, no selection', status='running', ckpt=None)
    print('recorded', RUN_ID)
print(f'n_train {n_train} | steps/epoch {steps_per_epoch} | planned {planned_steps}')

In [ ]:
#@title Train
import os, glob, importlib.util
os.environ['MODELSCOPE_CACHE']=str(MODEL_CACHE)
OUT_DIR.mkdir(parents=True, exist_ok=True)
if importlib.util.find_spec('bitsandbytes') is None:
    get_ipython().system('pip install -q -U bitsandbytes')

def find_ckpts(root):
    cs=glob.glob(str(root/'**'/'checkpoint-*'),recursive=True)
    cs=[x for x in cs if os.path.isdir(x) and x.rsplit('-',1)[1].isdigit()]
    return sorted(cs,key=lambda p:int(p.rsplit('-',1)[1]))
ckpts=find_ckpts(OUT_DIR); resume=ckpts[-1] if ckpts else None
print('checkpoints:',len(ckpts),'| resuming from:',resume or '(fresh start)')

attn = f' \\\n  --attn_impl {ATTN_IMPL}' if ATTN_IMPL else ''
cmd=f'''TQDM_MININTERVAL=30 MODELSCOPE_CACHE={MODEL_CACHE} swift sft \
  --model {MODEL_ID} \
  --model_type {MODEL_TYPE} \
  --tuner_type lora \
  --lora_rank {LORA_RANK} --lora_alpha {LORA_ALPHA} --target_modules all-linear \
  --dataset {train_jsonl} \
  --num_train_epochs {NUM_EPOCHS} \
  --per_device_train_batch_size {BATCH_SIZE} \
  --gradient_accumulation_steps {GRAD_ACCUM} \
  --learning_rate {LR} \
  --max_length {MAX_LENGTH} \
  --torch_dtype bfloat16 \
  --optim adamw_8bit \
  --save_steps {SAVE_STEPS} --save_total_limit {SAVE_LIMIT} \
  --logging_steps 5 \
  --output_dir {OUT_DIR} \
  --dataloader_num_workers 2 \
  --seed {SEED}''' + attn
if resume: cmd+=f' \\\n  --resume_from_checkpoint {resume}'
print(cmd)
if ALREADY_TRAINED:
    print('\nalready trained, skipping')
else:
    get_ipython().system(cmd)

In [ ]:
#@title Mark done
import glob, os
def find_ckpts(root):
    cs=glob.glob(str(root/'**'/'checkpoint-*'),recursive=True)
    cs=[x for x in cs if os.path.isdir(x) and x.rsplit('-',1)[1].isdigit()]
    return sorted(cs,key=lambda p:int(p.rsplit('-',1)[1]))
ckpts=find_ckpts(OUT_DIR); assert ckpts,'no checkpoint found'
if ALREADY_TRAINED: print('already marked done')
final=ckpts[-1]; steps=int(final.rsplit('-',1)[1])
print('final:',final); print(f'steps {steps} of {planned_steps}')
if ALREADY_TRAINED: pass
elif steps<planned_steps:
    print('incomplete: run the training cell again to resume')
else:
    rec_update(RUN_ID,status='done',ckpt=final,steps_actual=steps)
    print('marked done.')

In [ ]:
#@title Load the run record
rec=rec_load(RUN_ID)
assert rec, f'{RUN_ID} not recorded: run the training cells first'
assert rec.get('status')=='done', f"status {rec.get('status')}: training incomplete"
CKPT=rec['ckpt']; assert Path(CKPT).exists(), f'checkpoint missing: {CKPT}'
SPEC=json.load(open(DATA_DIR/'spec_v4.json')); TH=SPEC['thresholds']
print('ckpt  ',CKPT)
print('steps ',rec.get('steps_actual'),'of',rec.get('planned_steps'))

In [ ]:
#@title Evaluation subsets
import random
from collections import Counter
def load_jsonl(p):
    with open(p) as f: return [json.loads(l) for l in f if l.strip()]

CONDITIONS=['correct','contrastive','blind']
subset_paths={}; infer_paths={}
for cond in CONDITIONS:
    src=JSONL_DIR/f'eval_{cond}_{DATASET}_{MODALITY}_{TEMPLATE_ID}.jsonl'
    assert src.exists(), f'missing {src.name}'
    dst=SUBSET_DIR/f'sub_{cond}_{DATASET}_{MODALITY}_n{PER_CLASS}.jsonl'
    if dst.exists():
        rows=load_jsonl(dst); print(f'{cond:12} reuse n={len(rows)}')
    else:
        allr=load_jsonl(src); by={}
        for r in allr: by.setdefault(r['meta']['label'],[]).append(r)
        rng=random.Random(SUBSET_SEED); rows=[]
        for cls in sorted(by):
            pool=sorted(by[cls], key=lambda r: r['meta']['bearing']+r['messages'][1]['content'][:40])
            rng.shuffle(pool); rows+=pool[:PER_CLASS]
        rng.shuffle(rows)
        with open(dst,'w') as f:
            for r in rows: f.write(json.dumps(r)+'\n')
        print(f'{cond:12} drew  n={len(rows)}')
    subset_paths[cond]=dst
    q=dst.with_name(dst.stem+'_infer.jsonl')
    if not (q.exists() and sum(1 for _ in open(q)) == len(rows)):
        with open(q,'w') as f:
            for r in load_jsonl(dst):
                f.write(json.dumps({k:v for k,v in r.items() if k!='meta'})+'\n')
    infer_paths[cond]=q
    print(f'             labels {dict(Counter(r["meta"]["label"] for r in load_jsonl(dst)))}')

if RUN_SUBSTITUTION:
    import random as _rnd
    base=load_jsonl(subset_paths['blind'])
    n=len(base); perm=list(range(n)); _r=_rnd.Random(SUBST_SEED); _r.shuffle(perm)
    for i in range(n):
        if perm[i]==i:
            k=(i+1)%n; perm[i],perm[k]=perm[k],perm[i]
    assert all(perm[i]!=i for i in range(n)), 'not a derangement'
    dst=SUBSET_DIR/f'sub_substituted_{DATASET}_{MODALITY}_n{PER_CLASS}.jsonl'
    if not (dst.exists() and sum(1 for _ in open(dst)) == n):
      with open(dst,'w') as f:
        for i,r in enumerate(base):
            q=json.loads(json.dumps(r))
            src_row=base[perm[i]]
            if 'audios' in q: q['audios']=src_row['audios']
            if 'images' in q: q['images']=src_row['images']
            f.write(json.dumps(q)+'\n')
    _disk=load_jsonl(dst)
    assert all(_disk[i].get('audios')==base[perm[i]].get('audios') for i in range(n)), \
        'sub_substituted on disk differs from the SUBST_SEED derangement'
    subset_paths['substituted']=dst
    q=dst.with_name(dst.stem+'_infer.jsonl')
    if not (q.exists() and sum(1 for _ in open(q)) == n):
        with open(q,'w') as f:
            for r in load_jsonl(dst):
                f.write(json.dumps({k:v for k,v in r.items() if k!='meta'})+'\n')
    infer_paths['substituted']=q
    CONDITIONS=CONDITIONS+['substituted']
    print(f'substituted  n={n} (media deranged, text and labels unchanged)')
rec_update(RUN_ID, subsets={c:str(p) for c,p in subset_paths.items()})

In [ ]:
#@title Generate
import time, shutil
RESULT_DIR=EVAL_DIR/f'{RUN_ID}_{TEMPLATE_ID}'; RESULT_DIR.mkdir(parents=True,exist_ok=True)
print('results ->',RESULT_DIR)
def n_lines(p): return sum(1 for l in open(p) if l.strip()) if p.exists() else 0
result_paths={}
for cond in CONDITIONS:
    out=RESULT_DIR/f'infer_{cond}.jsonl'; result_paths[cond]=out
    need=n_lines(infer_paths[cond]); have=n_lines(out)
    if have==need:
        print(f'{cond}: complete ({have}/{need}), skipping'); continue
    if have:
        bak=out.with_suffix(f'.partial_{int(time.time())}.jsonl'); shutil.move(out,bak)
        print(f'{cond}: partial ({have}/{need}) moved to {bak.name}; regenerating')
    print(f'\n===== {cond} ({need} rows) ====='); t0=time.time()
    cmd=(f'TQDM_MININTERVAL=30 MODELSCOPE_CACHE={MODEL_CACHE} swift infer '
         f'--adapters {CKPT} --val_dataset {infer_paths[cond]} --result_path {out} '
         f'--max_new_tokens {MAX_NEW_TOKENS} --temperature 0 '
         f'--infer_backend transformers {INFER_EXTRA}'
         + (f' --attn_impl {ATTN_IMPL}' if ATTN_IMPL else ''))
    print(cmd); rc=os.system(cmd)
    have=n_lines(out)
    print(f'{cond}: exit {rc}, {have}/{need} rows, {(time.time()-t0)/60:.1f} min')
    if have!=need:
        raise RuntimeError(f'{cond}: swift infer wrote {have}/{need} rows (exit {rc})')

print('\nall conditions complete:')
for cond in CONDITIONS: print(f'   {cond:12} {n_lines(result_paths[cond])}/{n_lines(infer_paths[cond])}')
rec_update(RUN_ID, result_dir=str(RESULT_DIR), conditions=CONDITIONS)